# 01 – Explorácia datasetov

Notebook na prvotnú kontrolu pripravených datasetov **Cresci-17** a **Unmasking the Imposters**.

Pred spustením spusti `prepare_cresci.py` a `prepare_unmasking.py`. Notebook nič netrénuje – kontroluje štruktúru dát, počty záznamov, triedy, chýbajúce hodnoty, dĺžky textov a duplicity.


## 1. Import knižníc a nastavenie ciest

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Predpoklad: notebook je v priečinku notebooks/
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data" / "processed"
CRESCI_FILE = DATA_DIR / "cresci_prepared.csv"
UNMASKING_DIR = DATA_DIR / "unmasking-the-imposters"

print("Projektový priečinok:", PROJECT_DIR)
print("Cresci súbor existuje:", CRESCI_FILE.exists())
print("Unmasking priečinok existuje:", UNMASKING_DIR.exists())


## 2. Načítanie Cresci-17

In [ ]:
if not CRESCI_FILE.exists():
    raise FileNotFoundError(f"Nenašiel sa {CRESCI_FILE}. Najskôr spusti prepare_cresci.py.")

cresci = pd.read_csv(CRESCI_FILE, low_memory=False)
print("Rozmer datasetu:", cresci.shape)
print("Stĺpce:", cresci.columns.tolist())
display(cresci.head())


## 3. Rozdelenie tried v Cresci-17

In [ ]:
cresci_label_col = "label_name" if "label_name" in cresci.columns else "label"
cresci_counts = cresci[cresci_label_col].value_counts(dropna=False)
display(cresci_counts.rename("počet účtov").to_frame())

ax = cresci_counts.plot(kind="bar", figsize=(7, 4))
ax.set_title("Cresci-17 – počet účtov podľa triedy")
ax.set_xlabel("Trieda")
ax.set_ylabel("Počet účtov")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 4. Kontrola kategórií, chýbajúcich hodnôt a duplicít

In [ ]:
if "source" in cresci.columns:
    display(cresci["source"].value_counts(dropna=False).rename("počet").to_frame())

missing_cresci = cresci.isna().sum().sort_values(ascending=False).rename("počet_chýbajúcich").to_frame()
display(missing_cresci[missing_cresci["počet_chýbajúcich"] > 0])

if "user_id" in cresci.columns:
    print("Duplicitné user_id:", cresci["user_id"].duplicated().sum())


## 5. Dĺžka textov v Cresci-17

In [ ]:
if "text" not in cresci.columns:
    raise ValueError("V Cresci datasete chýba stĺpec text.")

cresci["text"] = cresci["text"].fillna("").astype(str)
cresci["text_length"] = cresci["text"].str.len()
display(cresci.groupby(cresci_label_col)["text_length"].describe().round(1))

cresci.boxplot(column="text_length", by=cresci_label_col, figsize=(8, 4))
plt.title("Dĺžka textu podľa triedy – Cresci-17")
plt.suptitle("")
plt.xlabel("Trieda")
plt.ylabel("Počet znakov")
plt.tight_layout()
plt.show()


## 6. Načítanie Unmasking the Imposters

In [ ]:
if not UNMASKING_DIR.exists():
    raise FileNotFoundError(f"Nenašiel sa {UNMASKING_DIR}. Najskôr spusti prepare_unmasking.py.")

unmasking_files = sorted(UNMASKING_DIR.glob("*/*.csv"))
unmasking_files = [p for p in unmasking_files if p.name in {"train.csv", "val.csv", "test.csv"}]

if not unmasking_files:
    raise FileNotFoundError("Nenašli sa CSV súbory v priečinkoch jednotlivých modelov.")

frames = []
for file in unmasking_files:
    part = pd.read_csv(file)
    if "model" not in part.columns:
        part["model"] = file.parent.name
    if "split" not in part.columns:
        part["split"] = file.stem
    frames.append(part)

unmasking = pd.concat(frames, ignore_index=True)
print("Počet načítaných súborov:", len(unmasking_files))
print("Rozmer datasetu:", unmasking.shape)
print("Modely:", sorted(unmasking["model"].dropna().unique()))
print("Splity:", sorted(unmasking["split"].dropna().unique()))
display(unmasking.head())


## 7. Počty textov podľa modelu, splitu a triedy

In [ ]:
unmasking_label_col = "label_name" if "label_name" in unmasking.columns else "label"
class_counts = (
    unmasking.groupby(["model", "split", unmasking_label_col])
    .size().rename("počet").reset_index()
)
display(class_counts)

pivot_counts = class_counts.pivot_table(
    index=["model", "split"],
    columns=unmasking_label_col,
    values="počet",
    fill_value=0
)
display(pivot_counts)


## 8. Graf rozdelenia tried podľa modelu

In [ ]:
model_class_counts = unmasking.groupby(["model", unmasking_label_col]).size().unstack(fill_value=0)
ax = model_class_counts.plot(kind="bar", figsize=(12, 5))
ax.set_title("Unmasking – počet ľudských a AI textov podľa modelu")
ax.set_xlabel("AI model")
ax.set_ylabel("Počet textov")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 9. Dĺžka textov v Unmasking

In [ ]:
unmasking["text"] = unmasking["text"].fillna("").astype(str)
unmasking["text_length"] = unmasking["text"].str.len()
display(unmasking.groupby(["model", unmasking_label_col])["text_length"].describe().round(1))

unmasking.boxplot(column="text_length", by=unmasking_label_col, figsize=(8, 4))
plt.title("Dĺžka textov podľa triedy – Unmasking")
plt.suptitle("")
plt.xlabel("Trieda")
plt.ylabel("Počet znakov")
plt.tight_layout()
plt.show()


## 10. Kontrola duplicít a prekrývania splitov

In [ ]:
print("Duplicitné texty celkovo:", unmasking["text"].duplicated().sum())

unique_texts = unmasking.groupby(["model", "split"])["text"].nunique().rename("unikátne_texty").to_frame()
display(unique_texts)

for model_name, group in unmasking.groupby("model"):
    split_sets = {
        split: set(part["text"].dropna())
        for split, part in group.groupby("split")
    }
    split_names = list(split_sets)
    for i in range(len(split_names)):
        for j in range(i + 1, len(split_names)):
            split_a, split_b = split_names[i], split_names[j]
            overlap = len(split_sets[split_a] & split_sets[split_b])
            if overlap:
                print("POZOR:", model_name, "-", overlap,
                      "rovnakých textov medzi splitmi", split_a, "a", split_b)


## 11. Zhrnutie kontroly

Po spustení si poznač:
- počet účtov v Cresci-17 a pomer bot/human,
- počet textov v Unmasking pre každý model a split,
- či sú triedy vyvážené,
- či sa vyskytujú prázdne alebo duplicitné texty,
- či sa rovnaké texty nachádzajú vo viacerých splitoch.

**Dôležité:** zatiaľ nič nevyvažuj ani automaticky nemaž. Najskôr vyhodnoť výsledky. Pri ďalších experimentoch zachováme pôvodné train/validation/test rozdelenie Unmasking; pri Cresci-17 vytvoríme vlastné rozdelenie na úrovni účtov.
